# Эксперименты: горизонт H и mult для triple-barrier

**Шаг 1 экспериментов.** Перебор H=3..7 и mult=1.0..2.0 для tb_vol. Сравнение AUC и net PnL (LightGBM).

Цель: найти комбинацию (H, mult), дающую лучшую точность и положительный PnL.

## 1. Импорты и загрузка

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

prepared_path = os.path.join(_root, 'outputs', 'prepared_with_rd_regime.parquet')
if not os.path.exists(prepared_path):
    raise FileNotFoundError(f'Запустите 01_Load_And_Prepare_Data.ipynb. Файл: {prepared_path}')

df = pd.read_parquet(prepared_path)
feat_path = os.path.join(_root, 'outputs', 'features_selected.txt')
scaler_path = os.path.join(_root, 'models', 'scaler.joblib')
if os.path.exists(feat_path):
    with open(feat_path) as f:
        FEATURES = [l.strip() for l in f if l.strip()]
elif os.path.exists(scaler_path):
    loaded = joblib.load(scaler_path)
    FEATURES = loaded.get('features', loaded.get('feature_names', []))
else:
    from src.features.feature_pipeline import get_feature_columns
    FEATURES = [c for c in get_feature_columns(include_symbol=True) if c in df.columns]
    FEATURES += [c for c in ['rd_regime', 'rd_regime_transition'] if c in df.columns]
FEATURES = [c for c in FEATURES if c in df.columns]
if len(FEATURES) < 5:
    FEATURES = [c for c in df.columns if c not in ['session_key', 'datetime', 'timestamp', 'rd_value', 'close_price', 'open', 'high', 'low', 'volume', 'source_file', 'source_day', 'time_diff_min'] and df[c].dtype in ['float64', 'int64']][:25]

print(f'Загружено: {len(df):,} строк. Фичей: {len(FEATURES)}')
print(f'LightGBM: {HAS_LGB}')

## 2. Функция triple-barrier и бектеста

In [ ]:
def add_tb_vol(df, by='session_key', vol_col='volatility_14', mult=1.5, h=5, out_col='tb'):
    """Triple-barrier: TP/SL = mult*vol, horizon=h. 1=TP, -1=SL, 0=time."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx = g.index
        c = g['close_price'].values
        h_arr = g['high'].values
        l_arr = g['low'].values
        v = g[vol_col].fillna(0.01).clip(lower=1e-6).values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - h):
            up = c[i] * (1 + mult * v[i])
            dn = c[i] * (1 - mult * v[i])
            for j in range(1, h + 1):
                if h_arr[i + j] >= up:
                    lab[i] = 1
                    break
                if l_arr[i + j] <= dn:
                    lab[i] = -1
                    break
            else:
                lab[i] = 0
        labels.loc[idx] = lab
    df[out_col] = labels.values
    return df

def backtest_pnl(proba, ret, commission=0.001, threshold=0.5):
    """pred=1 BUY, pred=0 SELL. threshold=None: trade every bar."""
    if threshold is None:
        pred = (proba >= 0.5).astype(int)
    else:
        pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    trade = np.where(pred == 1, 1, np.where(pred == 0, -1, 0))
    pnl_net = trade * ret - commission * (trade != 0)
    return {'trades': int((trade != 0).sum()), 'net_%': pnl_net.sum() * 100}

print('Функции готовы')

## 3. Перебор H и mult

In [ ]:
H_VALUES = [3, 4, 5, 6, 7]
MULT_VALUES = [1.0, 1.25, 1.5, 1.75, 2.0]
COMMISSION = 0.001
np.random.seed(42)

# Split по сессиям
sessions = list(df['session_key'].unique())
np.random.shuffle(sessions)
n = len(sessions)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_sessions = set(sessions[:n_train])
val_sessions = set(sessions[n_train:n_train + n_val])
test_sessions = set(sessions[n_train + n_val:])

df['ret_next'] = df.groupby('session_key')['close_price'].pct_change().shift(-1)
results = []

In [ ]:
for h in H_VALUES:
    for mult in MULT_VALUES:
        col = f'tb_h{h}_m{mult}'
        df_exp = add_tb_vol(df, mult=mult, h=h, out_col=col)
        valid = df_exp.dropna(subset=FEATURES + [col, 'ret_next']).copy()
        valid = valid[valid[col].isin([-1.0, 1.0])]
        if len(valid) < 1000:
            continue
        valid['y'] = (valid[col] == 1).astype(int)
        
        train_df = valid[valid['session_key'].isin(train_sessions)]
        test_df = valid[valid['session_key'].isin(test_sessions)]
        if len(train_df) < 500 or len(test_df) < 100:
            continue
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_df[FEATURES].fillna(0))
        X_test = scaler.transform(test_df[FEATURES].fillna(0))
        y_train = train_df['y'].values
        y_test = test_df['y'].values
        
        if not HAS_LGB:
            continue
        model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)
        
        ret = test_df['ret_next'].values
        r_full = backtest_pnl(proba, ret, commission=COMMISSION, threshold=None)
        r_06 = backtest_pnl(proba, ret, commission=COMMISSION, threshold=0.6)
        
        results.append({
            'h': h, 'mult': mult, 'auc': auc,
            'net_full_%': r_full['net_%'], 'trades_full': r_full['trades'],
            'net_06_%': r_06['net_%'], 'trades_06': r_06['trades'],
        })
        print(f'H={h} mult={mult}: AUC={auc:.4f} net_full={r_full["net_%"]:.2f}% net_06={r_06["net_%"]:.2f}%')

res_df = pd.DataFrame(results)

## 4. Сводная таблица и лучшие комбинации

In [ ]:
if len(res_df) > 0:
    print('Сводка (отсортировано по net_06_%):')
    print(res_df.sort_values('net_06_%', ascending=False).to_string(index=False))
    
    best_row = res_df.loc[res_df['net_06_%'].idxmax()]
    print(f"\nЛучшая комбинация: H={best_row['h']}, mult={best_row['mult']}")
    print(f"  AUC: {best_row['auc']:.4f}, net (proba>0.6): {best_row['net_06_%']:.2f}%")
    
    best_auc = res_df.loc[res_df['auc'].idxmax()]
    print(f"\nЛучшая AUC: H={best_auc['h']}, mult={best_auc['mult']} -> {best_auc['auc']:.4f}")

## 5. Сохранение лучшей конфигурации для следующих шагов

In [ ]:
# Сохраняем выводы для следующих ноутбуков
out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)
if len(res_df) > 0:
    res_df.to_csv(os.path.join(out_dir, 'exp10_horizon_mult_results.csv'), index=False)
    best = res_df.loc[res_df['net_06_%'].idxmax()]
    best_config = {'h': int(best['h']), 'mult': float(best['mult'])}
    joblib.dump(best_config, os.path.join(out_dir, 'exp10_best_horizon_mult.joblib'))
    print(f"Сохранено: exp10_horizon_mult_results.csv, exp10_best_horizon_mult.joblib")
    print(f"Best config: {best_config}")

## Итог шага 10

- Перебраны H=3..7 и mult=1.0..2.0 для triple-barrier.
- LightGBM: AUC и net PnL (full/proba>0.6) для каждой комбинации.
- Лучшая конфигурация сохранена в outputs/exp10_best_horizon_mult.joblib.

# Эксперименты: горизонт H и mult для triple-barrier

**Шаг 1 плана 06_EXPERIMENTS_PLAN.md.** Тестирование различных H (3–7) и mult (1.0–2.0) для target tb_vol. Цель: найти комбинацию с лучшим AUC и положительным net PnL.

## 1. Импорты и загрузка

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

prepared_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
if not os.path.exists(prepared_path):
    raise FileNotFoundError(f'Запустите 04_Data_Labeling_And_Feature_Loading.ipynb. Файл: {prepared_path}')

df = pd.read_parquet(prepared_path)
print(f'Загружено: {len(df):,} строк, {df["session_key"].nunique()} сессий')

## 2. Функция add_tb_vol (универсальная)

In [ ]:
def add_tb_vol(df, by='session_key', vol_col='volatility_14', mult=1.5, h=5, col_name=None):
    """Triple-barrier: TP/SL = mult*vol, horizon=h. 1=TP, -1=SL, 0=time."""
    df = df.copy()
    labels = pd.Series(index=df.index, dtype=np.float64)
    for skey, g in df.groupby(by, sort=False):
        idx = g.index
        c = g['close_price'].values
        h_arr = g['high'].values
        l_arr = g['low'].values
        v = g[vol_col].fillna(0.01).clip(lower=1e-6).values
        lab = np.full(len(g), np.nan, dtype=np.float64)
        for i in range(len(g) - h):
            up = c[i] * (1 + mult * v[i])
            dn = c[i] * (1 - mult * v[i])
            for j in range(1, h + 1):
                if h_arr[i + j] >= up:
                    lab[i] = 1
                    break
                if l_arr[i + j] <= dn:
                    lab[i] = -1
                    break
            else:
                lab[i] = 0
        labels.loc[idx] = lab
    name = col_name or f'tb_vol_h{h}_m{mult}'
    df[name] = labels.values
    return df, name

## 3. Сетка экспериментов (H, mult)

In [ ]:
H_VALUES = [3, 4, 5, 6, 7]
MULT_VALUES = [1.0, 1.25, 1.5, 1.75, 2.0]

results_grid = []
for h in H_VALUES:
    for mult in MULT_VALUES:
        df_tmp, col = add_tb_vol(df, mult=mult, h=h)
        vc = df_tmp[col].dropna().value_counts()
        n_buy = vc.get(1.0, 0)
        n_sell = vc.get(-1.0, 0)
        n_hold = vc.get(0.0, 0)
        total = n_buy + n_sell + n_hold
        results_grid.append({
            'h': h,
            'mult': mult,
            'col': col,
            'n_buy': n_buy,
            'n_sell': n_sell,
            'n_hold': n_hold,
            'buy_pct': n_buy / total if total else 0,
            'sell_pct': n_sell / total if total else 0,
            'hold_pct': n_hold / total if total else 0,
        })

grid_df = pd.DataFrame(results_grid)
print('Распределения по H и mult:')
print(grid_df[['h', 'mult', 'buy_pct', 'sell_pct', 'hold_pct']].to_string(index=False))

## 4. Наивный PnL для каждой конфигурации

In [ ]:
df['ret_next'] = df.groupby('session_key')['close_price'].pct_change().shift(-1)
COMMISSION = 0.001

def naive_pnl(df_base, h, mult):
    d, col = add_tb_vol(df_base.copy(), mult=mult, h=h)
    v = d.dropna(subset=[col, 'ret_next'])
    v = v[v[col].isin([-1.0, 1.0])]
    if len(v) == 0:
        return 0.0, 0
    trade = v[col].values
    ret = v['ret_next'].values
    pnl_raw = (trade * ret).sum() * 100
    n_trades = len(v)
    pnl_net = pnl_raw - n_trades * COMMISSION * 100
    return pnl_net, n_trades

for r in results_grid:
    pnl, n = naive_pnl(df, r['h'], r['mult'])
    r['naive_net_pnl'] = pnl
    r['naive_trades'] = n

grid_df = pd.DataFrame(results_grid)
best_naive = grid_df.loc[grid_df['naive_net_pnl'].idxmax()]
print('Лучшая конфигурация по наивному PnL:')
print(best_naive[['h', 'mult', 'naive_net_pnl', 'naive_trades']].to_string())
print('\nТоп-5:')
print(grid_df.nlargest(5, 'naive_net_pnl')[['h', 'mult', 'naive_net_pnl', 'naive_trades']].to_string(index=False))

## 5. LightGBM + backtest для каждой конфигурации

In [ ]:
try:
    import lightgbm as lgb
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

FEATURES_BASE = [c for c in get_feature_columns(include_symbol=True) 
                 if c in df.columns and c not in ['rd_mom_1']]
if 'rd_regime' not in FEATURES_BASE and 'rd_regime' in df.columns:
    FEATURES_BASE.append('rd_regime')
if 'rd_regime_transition' not in FEATURES_BASE and 'rd_regime_transition' in df.columns:
    FEATURES_BASE.append('rd_regime_transition')

print(f'Фичей: {len(FEATURES_BASE)}')
print('LightGBM:', 'OK' if HAS_LGB else 'НЕ УСТАНОВЛЕН')

In [ ]:
def train_and_backtest(df_base, h, mult, features, commission=0.001, seed=42):
    """Обучает LightGBM, возвращает AUC и net PnL (proba>0.6)."""
    d, col = add_tb_vol(df_base.copy(), mult=mult, h=h)
    valid = d.dropna(subset=features + [col]).copy()
    valid = valid[valid[col].isin([-1.0, 1.0])]
    valid['y'] = (valid[col] == 1).astype(int)
    
    sessions = valid['session_key'].unique().tolist()
    np.random.seed(seed)
    np.random.shuffle(sessions)
    n = len(sessions)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)
    train_sessions = sessions[:n_train]
    test_sessions = sessions[n_train + n_val:]
    
    train_df = valid[valid['session_key'].isin(train_sessions)]
    test_df = valid[valid['session_key'].isin(test_sessions)]
    
    X_train = train_df[features].fillna(0)
    X_test = test_df[features].fillna(0)
    y_train = train_df['y'].values
    y_test = test_df['y'].values
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=seed, verbose=-1)
    model.fit(X_train_s, y_train)
    
    proba = model.predict_proba(X_test_s)[:, 1]
    auc = roc_auc_score(y_test, proba)
    
    test_df = test_df.copy()
    test_df['ret_next'] = test_df.groupby('session_key')['close_price'].pct_change().shift(-1)
    bt = test_df.dropna(subset=['ret_next'])
    if len(bt) == 0:
        return auc, 0.0, 0
    
    X_bt = scaler.transform(bt[features].fillna(0))
    proba_bt = model.predict_proba(X_bt)[:, 1]
    ret = bt['ret_next'].values
    
    trade = np.where(proba_bt >= 0.6, 1, np.where(proba_bt <= 0.4, -1, 0))
    pnl_net = (trade * ret).sum() * 100 - (trade != 0).sum() * commission * 100
    n_trades = (trade != 0).sum()
    
    return auc, pnl_net, n_trades

if HAS_LGB:
    ml_results = []
    for r in results_grid:
        auc, pnl, n = train_and_backtest(df, r['h'], r['mult'], FEATURES_BASE)
        r['auc'] = auc
        r['model_net_pnl'] = pnl
        r['model_trades'] = n
        ml_results.append((r['h'], r['mult'], auc, pnl))
    
    ml_df = pd.DataFrame(results_grid)
    best_ml = ml_df.loc[ml_df['model_net_pnl'].idxmax()]
    print('Лучшая конфигурация по net PnL модели (proba>0.6):')
    print(f"  H={best_ml['h']}, mult={best_ml['mult']}, AUC={best_ml['auc']:.4f}, net PnL={best_ml['model_net_pnl']:.2f}%, trades={best_ml['model_trades']}")
    print('\nТоп-5 по model_net_pnl:')
    print(ml_df.nlargest(5, 'model_net_pnl')[['h', 'mult', 'auc', 'model_net_pnl', 'model_trades']].to_string(index=False))

## 6. Итоговая сводная таблица

In [ ]:
summary = pd.DataFrame(results_grid)
out_path = os.path.join(os.getcwd(), 'outputs', '10_horizon_mult_results.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
summary.to_csv(out_path, index=False)
print(f'Сохранено: {out_path}')
print('\nСводка (все конфигурации):')
cols_show = ['h', 'mult', 'naive_net_pnl', 'model_net_pnl', 'auc'] if 'auc' in summary.columns else ['h', 'mult', 'naive_net_pnl']
print(summary[cols_show].to_string(index=False))

## Итог шага 1

- Протестированы H=3..7 и mult=1.0..2.0
- Результаты сохранены в `outputs/10_horizon_mult_results.csv`
- Выбрать лучшую (H, mult) для следующих экспериментов